# NB12 — Clustered Uncertainty and Egg-Influence Analysis | NIR-HUEVOS 2026

This notebook adds the CNN1D to the existing six-model benchmark and performs inference at the **egg level (N = 30)**. It does not retrain models and does not treat 660 longitudinal spectra as independent biological replicates.

Outputs include:

- pooled and per-egg metrics for seven models;
- 10,000 egg-cluster bootstrap confidence intervals;
- paired bootstrap confidence intervals for MAE differences;
- Friedman/Kendall-W and paired Wilcoxon tests with Holm correction;
- leave-one-egg-out influence diagnostics on fixed OOF predictions;
- outer-fold heterogeneity summaries.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import hashlib, json, math, platform, sys, warnings
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, rankdata, wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
R03 = PROJECT_ROOT / '05_RESULTS' / 'NB03_CHEMOMETRIC_BASELINES'
R04 = PROJECT_ROOT / '05_RESULTS' / 'NB04_DEEP_LEARNING_BENCHMARK'
ROUND = PROJECT_ROOT / '05_RESULTS' / 'REVISION_REVIEWERS_2026_09'
R11 = ROUND / 'NB11_CNN1D_REVIEWER_BENCHMARK'
RESULT_DIR = ROUND / 'NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

P03 = R03 / 'NB03_oof_predictions.csv'
P04 = R04 / 'NB04_oof_predictions_seedmean.csv'
P11 = R11 / 'NB11_oof_predictions_seedmean.csv'
S11 = R11 / 'EXECUTION_STATUS.json'
for p in [P03, P04, P11, S11]: assert p.exists(), f'Missing prerequisite: {p}'
assert json.loads(S11.read_text())['status'] == 'SUCCESS', 'NB11 is not complete.'

MODELS = ['SVR','PLSR','ANN','SimpleRNN','LSTM','BiLSTM','CNN1D']
BOOTSTRAP_REPS = 10000
BOOTSTRAP_SEED = 20260915
ALPHA = 0.05
print('Prerequisites found. Statistical unit: egg (N=30).')


Prerequisites found. Statistical unit: egg (N=30).


In [3]:
# Harmonize saved OOF predictions without recalculating or altering them
def standardize(frame, source):
    frame = frame.copy()
    rename = {}
    for c in frame.columns:
        low = c.lower()
        if low in ['prediction','predicted','yhat','y_pred_mean','prediction_seedmean']:
            rename[c] = 'y_pred'
        elif low in ['y_true','observed','target','storage_day']:
            rename[c] = 'storage_days'
    frame = frame.rename(columns=rename)
    required = {'sample','storage_days','model','y_pred'}
    assert required.issubset(frame.columns), f'{source}: columns={frame.columns.tolist()}'
    keep = ['sample','storage_days','model','y_pred'] + (['outer_fold'] if 'outer_fold' in frame.columns else [])
    frame = frame[keep]
    frame['model'] = frame['model'].replace({'CNN1D':'CNN1D'})
    frame['source'] = source
    return frame

p03 = standardize(pd.read_csv(P03), 'NB03')
p04 = standardize(pd.read_csv(P04), 'NB04')
p11 = standardize(pd.read_csv(P11), 'NB11')
oof = pd.concat([p03, p04, p11], ignore_index=True)
oof = oof[oof.model.isin(MODELS)].copy()
assert set(oof.model) == set(MODELS), sorted(oof.model.unique())
assert oof.groupby('model').size().eq(660).all()
assert not oof.duplicated(['model','sample','storage_days']).any()
assert oof.groupby('model')['sample'].nunique().eq(30).all()
assert oof.groupby('model')['storage_days'].nunique().eq(22).all()
oof['abs_error'] = (oof.y_pred - oof.storage_days).abs()
oof['sq_error'] = (oof.y_pred - oof.storage_days)**2
oof.to_csv(RESULT_DIR / 'NB12_unified_oof_predictions.csv', index=False)
print(oof.groupby('model').size())


model
ANN          660
BiLSTM       660
CNN1D        660
LSTM         660
PLSR         660
SVR          660
SimpleRNN    660
dtype: int64


In [4]:
# Descriptive performance and per-egg error matrix
def perf(g):
    return pd.Series({
        'MAE_days': mean_absolute_error(g.storage_days, g.y_pred),
        'RMSE_days': np.sqrt(mean_squared_error(g.storage_days, g.y_pred)),
        'R2': r2_score(g.storage_days, g.y_pred),
        'bias_days': np.mean(g.y_pred - g.storage_days),
        'median_AE_days': np.median(np.abs(g.y_pred - g.storage_days))
    })

pooled = oof.groupby('model', sort=False).apply(perf).reset_index()
egg_long = oof.groupby(['model','sample'], as_index=False).agg(
    MAE_days=('abs_error','mean'), RMSE_days=('sq_error', lambda x: np.sqrt(x.mean())),
    bias_days=('y_pred', lambda x: np.nan)
)
# Bias is computed explicitly to avoid an ambiguous aggregation closure.
bias = oof.assign(error=oof.y_pred-oof.storage_days).groupby(['model','sample'],as_index=False).error.mean()
egg_long = egg_long.drop(columns='bias_days').merge(bias.rename(columns={'error':'bias_days'}), on=['model','sample'])
egg_wide = egg_long.pivot(index='sample', columns='model', values='MAE_days').reindex(columns=MODELS)
assert egg_wide.shape == (30,7) and not egg_wide.isna().any().any()
pooled.to_csv(RESULT_DIR / 'NB12_pooled_metrics_7models.csv', index=False)
egg_long.to_csv(RESULT_DIR / 'NB12_per_egg_metrics_long.csv', index=False)
egg_wide.reset_index().to_csv(RESULT_DIR / 'NB12_per_egg_MAE_wide.csv', index=False)
display(pooled.sort_values('MAE_days'))


,model,MAE_days,RMSE_days,R2,bias_days,median_AE_days
1,SVR,2.194628,2.716171,0.816706,0.147892,1.954567
0,PLSR,2.267298,2.863691,0.796255,0.006188,1.869861
2,ANN,2.288940,2.974129,0.780237,-0.158703,1.835875
3,BiLSTM,4.558494,5.586267,0.224686,-0.427800,4.152189
6,CNN1D,4.571915,5.623247,0.214387,0.022471,4.001830
4,LSTM,4.709926,5.571621,0.228746,-0.224954,4.445259
5,SimpleRNN,4.878542,5.721820,0.186603,0.173950,4.790998


In [5]:
# Egg-cluster bootstrap: vectorized resampling of whole 22-day egg trajectories
rng = np.random.default_rng(BOOTSTRAP_SEED)
egg_ids = np.array(sorted(oof['sample'].unique()))
target_days = np.array(sorted(oof['storage_days'].unique()), dtype=float)
pred_tensor = np.empty((len(MODELS), len(egg_ids), len(target_days)), dtype=np.float64)
for mi, model in enumerate(MODELS):
    pivot = (oof[oof.model == model]
             .pivot(index='sample', columns='storage_days', values='y_pred')
             .reindex(index=egg_ids, columns=target_days.astype(int)))
    assert not pivot.isna().any().any()
    pred_tensor[mi] = pivot.to_numpy(dtype=float)
error_tensor = pred_tensor - target_days[None, None, :]
sample_idx = rng.integers(0, len(egg_ids), size=(BOOTSTRAP_REPS, len(egg_ids)))
mae_boot = np.empty((len(MODELS), BOOTSTRAP_REPS), dtype=float)
rmse_boot = np.empty_like(mae_boot)
r2_boot = np.empty_like(mae_boot)
tss_per_replicate = len(egg_ids) * np.sum((target_days - target_days.mean())**2)
chunk = 250
for start in range(0, BOOTSTRAP_REPS, chunk):
    stop = min(start + chunk, BOOTSTRAP_REPS)
    sampled_error = error_tensor[:, sample_idx[start:stop], :]
    mae_boot[:, start:stop] = np.mean(np.abs(sampled_error), axis=(2,3))
    mse = np.mean(sampled_error**2, axis=(2,3))
    rmse_boot[:, start:stop] = np.sqrt(mse)
    sse = np.sum(sampled_error**2, axis=(2,3))
    r2_boot[:, start:stop] = 1.0 - sse/tss_per_replicate

boot_metrics = pd.concat([
    pd.DataFrame({'replicate':np.arange(BOOTSTRAP_REPS),'model':model,
                  'MAE_days':mae_boot[mi],'RMSE_days':rmse_boot[mi],'R2':r2_boot[mi]})
    for mi,model in enumerate(MODELS)
],ignore_index=True)
boot_diffs = pd.concat([
    pd.DataFrame({'replicate':np.arange(BOOTSTRAP_REPS),'model_a':a,'model_b':b,
                  'delta_MAE_a_minus_b':mae_boot[MODELS.index(a)]-mae_boot[MODELS.index(b)]})
    for a,b in combinations(MODELS,2)
],ignore_index=True)
ci_rows = []
for model, g in boot_metrics.groupby('model'):
    point = pooled.loc[pooled.model==model].iloc[0]
    ci_rows.append({
        'model':model,
        'MAE_days':point.MAE_days,
        'MAE_CI_low':g.MAE_days.quantile(ALPHA/2),'MAE_CI_high':g.MAE_days.quantile(1-ALPHA/2),
        'RMSE_days':point.RMSE_days,
        'RMSE_CI_low':g.RMSE_days.quantile(ALPHA/2),'RMSE_CI_high':g.RMSE_days.quantile(1-ALPHA/2),
        'R2':point.R2,'R2_CI_low':g.R2.quantile(ALPHA/2),'R2_CI_high':g.R2.quantile(1-ALPHA/2)
    })
metric_ci = pd.DataFrame(ci_rows).sort_values('MAE_days')
diff_ci = boot_diffs.groupby(['model_a','model_b'],as_index=False).agg(
    delta_MAE_mean=('delta_MAE_a_minus_b','mean'),
    delta_MAE_CI_low=('delta_MAE_a_minus_b',lambda x:x.quantile(ALPHA/2)),
    delta_MAE_CI_high=('delta_MAE_a_minus_b',lambda x:x.quantile(1-ALPHA/2)),
    probability_a_better=('delta_MAE_a_minus_b',lambda x:float(np.mean(x<0)))
)
metric_ci.to_csv(RESULT_DIR / 'NB12_cluster_bootstrap_metric_CI.csv', index=False)
diff_ci.to_csv(RESULT_DIR / 'NB12_paired_bootstrap_MAE_differences.csv', index=False)
# Replicate-level bootstrap data are retained for independent verification.
boot_metrics.to_csv(RESULT_DIR / 'NB12_cluster_bootstrap_metric_replicates.csv.gz', index=False, compression='gzip')
boot_diffs.to_csv(RESULT_DIR / 'NB12_paired_bootstrap_difference_replicates.csv.gz', index=False, compression='gzip')
display(metric_ci)


,model,MAE_days,MAE_CI_low,MAE_CI_high,RMSE_days,RMSE_CI_low,RMSE_CI_high,R2,R2_CI_low,R2_CI_high
5,SVR,2.194628,2.027259,2.370625,2.716171,2.527328,2.911887,0.816706,0.789339,0.841307
4,PLSR,2.267298,2.083677,2.490164,2.863691,2.619288,3.141824,0.796255,0.754756,0.829549
0,ANN,2.288940,2.096045,2.519131,2.974129,2.722894,3.255320,0.780237,0.736718,0.815798
1,BiLSTM,4.558494,4.228119,4.895744,5.586267,5.125744,6.036512,0.224686,0.094671,0.347248
2,CNN1D,4.571915,4.092267,5.088248,5.623247,5.059596,6.197513,0.214387,0.045735,0.363987
3,LSTM,4.709926,4.516949,4.910556,5.571621,5.369203,5.776828,0.228746,0.170888,0.283768
6,SimpleRNN,4.878542,4.672197,5.079178,5.721820,5.481223,5.956245,0.186603,0.118588,0.253570


In [6]:
# Friedman omnibus test, Kendall W, and paired Wilcoxon tests with Holm correction
arrays = [egg_wide[m].to_numpy() for m in MODELS]
friedman_stat, friedman_p = friedmanchisquare(*arrays)
n, k = egg_wide.shape
kendall_w = friedman_stat / (n*(k-1))
friedman = pd.DataFrame([{'n_eggs':n,'n_models':k,'chi_square':friedman_stat,'df':k-1,'p_value':friedman_p,'kendall_W':kendall_w}])

def rank_biserial(d):
    d = np.asarray(d, float); d = d[d != 0]
    ranks = rankdata(np.abs(d))
    return float((ranks[d>0].sum() - ranks[d<0].sum()) / ranks.sum()) if len(d) else 0.0

tests = []
for a,b in combinations(MODELS,2):
    d = egg_wide[a].to_numpy() - egg_wide[b].to_numpy()
    stat,p = wilcoxon(d, alternative='two-sided', zero_method='wilcox', method='auto')
    tests.append({
        'model_a':a,'model_b':b,'mean_delta_MAE_a_minus_b':d.mean(),
        'median_delta_MAE_a_minus_b':np.median(d),'wilcoxon_statistic':stat,
        'p_raw':p,'rank_biserial_positive_means_a_worse':rank_biserial(d)
    })
tests = pd.DataFrame(tests).sort_values('p_raw').reset_index(drop=True)
m = len(tests)
adj = np.maximum.accumulate(np.minimum(1.0, tests.p_raw.to_numpy()*(m-np.arange(m))))
tests['p_holm'] = adj
tests['reject_holm_0_05'] = tests.p_holm < 0.05
friedman.to_csv(RESULT_DIR / 'NB12_friedman_kendall.csv', index=False)
tests.to_csv(RESULT_DIR / 'NB12_pairwise_wilcoxon_holm.csv', index=False)
display(friedman); display(tests)


,n_eggs,n_models,chi_square,df,p_value,kendall_W
0,30,7,138.528571,6,2.048473e-27,0.769603


,model_a,model_b,mean_delta_MAE_a_minus_b,median_delta_MAE_a_minus_b,wilcoxon_statistic,p_raw,rank_biserial_positive_means_a_worse,p_holm,reject_holm_0_05
0,SVR,LSTM,-2.515298,-2.556946,0.0,1.862645e-09,-1.000000,3.911555e-08,True
1,SVR,SimpleRNN,-2.683915,-2.678046,0.0,1.862645e-09,-1.000000,3.911555e-08,True
2,SVR,CNN1D,-2.377287,-2.256889,0.0,1.862645e-09,-1.000000,3.911555e-08,True
3,SVR,BiLSTM,-2.363867,-2.370779,0.0,1.862645e-09,-1.000000,3.911555e-08,True
4,PLSR,SimpleRNN,-2.611244,-2.618138,0.0,1.862645e-09,-1.000000,3.911555e-08,True
5,PLSR,CNN1D,-2.304617,-2.108362,0.0,1.862645e-09,-1.000000,3.911555e-08,True
6,PLSR,BiLSTM,-2.291196,-2.263982,0.0,1.862645e-09,-1.000000,3.911555e-08,True
7,PLSR,LSTM,-2.442627,-2.470281,0.0,1.862645e-09,-1.000000,3.911555e-08,True
8,ANN,LSTM,-2.420986,-2.528740,0.0,1.862645e-09,-1.000000,3.911555e-08,True
9,ANN,BiLSTM,-2.269554,-2.185376,0.0,1.862645e-09,-1.000000,3.911555e-08,True


In [7]:
# Leave-one-egg-out influence on fixed OOF performance estimates (no model refitting)
influence_rows = []
for model,g in oof.groupby('model'):
    full = perf(g)
    for egg in egg_ids:
        reduced = g[g['sample'] != egg]
        p = perf(reduced)
        influence_rows.append({
            'model':model,'omitted_egg':int(egg),
            'MAE_without_egg':p.MAE_days,'delta_MAE_without_minus_full':p.MAE_days-full.MAE_days,
            'RMSE_without_egg':p.RMSE_days,'delta_RMSE_without_minus_full':p.RMSE_days-full.RMSE_days,
            'R2_without_egg':p.R2,'delta_R2_without_minus_full':p.R2-full.R2
        })
influence = pd.DataFrame(influence_rows)
influence_summary = influence.groupby('model',as_index=False).agg(
    max_abs_delta_MAE=('delta_MAE_without_minus_full',lambda x:np.max(np.abs(x))),
    max_abs_delta_RMSE=('delta_RMSE_without_minus_full',lambda x:np.max(np.abs(x))),
    max_abs_delta_R2=('delta_R2_without_minus_full',lambda x:np.max(np.abs(x))),
    most_influential_egg_MAE=('delta_MAE_without_minus_full',lambda x:int(influence.loc[x.abs().idxmax(),'omitted_egg']))
)
influence.to_csv(RESULT_DIR / 'NB12_leave_one_egg_out_influence.csv', index=False)
influence_summary.to_csv(RESULT_DIR / 'NB12_leave_one_egg_out_summary.csv', index=False)
display(influence_summary)


,model,max_abs_delta_MAE,max_abs_delta_RMSE,max_abs_delta_R2,most_influential_egg_MAE
0,ANN,0.076392,0.102574,0.014897,17
1,BiLSTM,0.054351,0.094277,0.025948,2
2,CNN1D,0.153327,0.188181,0.051701,17
3,LSTM,0.058052,0.060495,0.016657,20
4,PLSR,0.079533,0.109123,0.015232,17
5,SVR,0.039783,0.047594,0.006367,17
6,SimpleRNN,0.036061,0.048550,0.013745,4


In [8]:
# Outer-fold heterogeneity reconstructed from frozen OOF assignments
fold_rows=[]
for (model,fold),g in oof.groupby(['model','outer_fold']):
    row={'model':model,'outer_fold':int(fold),'n_eggs':g['sample'].nunique(),'n_spectra':len(g)}
    row.update(perf(g).to_dict()); fold_rows.append(row)
fold_metrics=pd.DataFrame(fold_rows)
fold_summary=fold_metrics.groupby('model',as_index=False).agg(
    mean_fold_MAE=('MAE_days','mean'),sd_fold_MAE=('MAE_days','std'),
    min_fold_MAE=('MAE_days','min'),max_fold_MAE=('MAE_days','max'),
    mean_fold_R2=('R2','mean'),sd_fold_R2=('R2','std')
)
fold_metrics.to_csv(RESULT_DIR/'NB12_outer_fold_metrics_7models.csv',index=False)
fold_summary.to_csv(RESULT_DIR/'NB12_outer_fold_heterogeneity.csv',index=False)
display(fold_summary.sort_values('mean_fold_MAE'))


,model,mean_fold_MAE,sd_fold_MAE,min_fold_MAE,max_fold_MAE,mean_fold_R2,sd_fold_R2
5,SVR,2.194628,0.196170,2.060998,2.537884,0.816706,0.039201
4,PLSR,2.267298,0.314654,2.028061,2.781110,0.796255,0.064042
0,ANN,2.288940,0.313474,1.961318,2.755875,0.780237,0.062457
1,BiLSTM,4.558494,0.261073,4.111605,4.795141,0.224686,0.130902
2,CNN1D,4.571915,0.570438,4.036906,5.475258,0.214387,0.187072
3,LSTM,4.709926,0.207746,4.412628,4.921388,0.228746,0.070016
6,SimpleRNN,4.878542,0.235291,4.553500,5.158409,0.186603,0.079163


In [9]:
# Protocol and completion status
protocol={
    'notebook':'NB12_CLUSTERED_UNCERTAINTY_AND_INFLUENCE.ipynb',
    'analysis_role':'additional reviewer inference using frozen OOF predictions',
    'statistical_unit':'egg','n_independent_eggs':30,'n_longitudinal_spectra':660,
    'models':MODELS,'bootstrap_replicates':BOOTSTRAP_REPS,'bootstrap_seed':BOOTSTRAP_SEED,
    'bootstrap_scheme':'resample eggs with replacement; retain all 22 rows within each sampled egg',
    'omnibus_test':'Friedman on paired per-egg MAE','effect_size':'Kendall W',
    'pairwise_test':'two-sided paired Wilcoxon on per-egg MAE','multiplicity':'Holm within 21 model pairs',
    'influence_analysis':'delete one egg from fixed OOF predictions; no retraining',
    'equivalence_claimed':False
}
(RESULT_DIR/'NB12_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
status={
    'notebook':protocol['notebook'],'status':'SUCCESS','completed_at_utc':datetime.now(timezone.utc).isoformat(),
    'n_models':len(MODELS),'n_eggs':30,'bootstrap_replicates':BOOTSTRAP_REPS,
    'pairwise_comparisons':len(tests),'all_models_complete':True
}
(RESULT_DIR/'EXECUTION_STATUS.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('NB12 SUCCESS. Next: NB13_REVIEWER_TABLES_FIGURES.ipynb')


NB12 SUCCESS. Next: NB13_REVIEWER_TABLES_FIGURES.ipynb
